# Keras/TensorFlow — Chapter 12: Three Ways to Build Models in Keras


## 1. Sequential 

- Cú pháp: liệt kê layer trong `Sequential([...])`. Chỉ dùng được khi mỗi layer có **đúng 1 input, 1 output**.
- Ví dụ: LeNet-5 áp dụng cho **CIFAR-10** (ảnh màu 32×32×3, khác MNIST xám 28×28×1) — 697,046 tham số.
- **Kết quả**: train accuracy tăng đều 36%→82% qua 10 epoch, nhưng `val_loss` lại **tăng dần** sau vài epoch đầu (từ ~1.0 lên 1.63) — dấu hiệu overfitting kinh điển: model học thuộc train tốt hơn nhưng khái quát hoá kém đi.

## 2. Functional API

- Mỗi layer được gọi như một **hàm nhận tensor, trả về tensor**: `x = Conv2D(...)(x)` — thay vì thêm vào một danh sách tuần tự.
- Cùng kiến trúc LeNet-5 viết bằng Functional API cho **đúng 697,046 tham số** như bản Sequential — xác nhận 2 cách viết chỉ khác cú pháp, không khác kiến trúc.
- **Trường hợp Sequential không làm được**: khối **residual (ResNet)** — có **skip connection** (cộng thẳng input vào output của 2 lớp conv sau). Functional API giải quyết bằng layer `Add()([identity, x])` nhận 2 input.
- Model với residual block (1.17 triệu tham số) cho **val_loss ổn định hơn** (dao động quanh 1.0–1.2) so với LeNet-5 thường (tăng dần lên 1.63), đây là lợi ích của skip connection trong việc giảm overfitting.

## 3. Subclassing keras.Model 

- Định nghĩa layer trong `__init__()`, định nghĩa luồng dữ liệu trong `call()`.
- **Lưu ý**: layer phải tạo trong `__init__()`, **không được tạo trong `call()`** — nếu không sẽ gặp lỗi `tf.Variable can only be created once`, vì `call()` được gọi lại nhiều lần nhưng object layer phải giữ nguyên để tối ưu đúng trọng số qua các lần gọi.

## 4. Khi nào dùng cách nào

| Cách viết | Ưu điểm | Hạn chế |
|---|---|---|
| Sequential | Ngắn gọn nhất | Không phân nhánh được (1 input/1 output mỗi layer) |
| Functional | Linh hoạt: đa nhánh, skip connection, đa input | Dài hơn Sequential |
| Subclassing | Hướng đối tượng, dễ tái sử dụng | Dài nhất, dễ mắc lỗi tạo layer sai chỗ |


## 5. Vận dụng


**12.1–12.2** — LeNet-5 bằng Sequential class

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D, MaxPool2D

model = Sequential([
          Input(shape=(32,32,3,)),
          Conv2D(6, (5,5), padding="same", activation="relu"),
          MaxPool2D(pool_size=(2,2)),
          Conv2D(16, (5,5), padding="same", activation="relu"),
          MaxPool2D(pool_size=(2, 2)),
          Conv2D(120, (5,5), padding="same", activation="relu"),
          Flatten(),
          Dense(units=84, activation="relu"),
          Dense(units=10, activation="softmax"),
      ])

model.summary()


2026-09-06 07:15:31.284989: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:15:31.358135: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 07:15:31.358173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 07:15:31.359637: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 07:15:31.378215: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:15:31.379787: I tensorflow/core/platform/cpu_feature_guard.cc:1

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 6)         456       
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 6)         0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 16)        2416      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 8, 8, 16)          0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 8, 8, 120)         48120     
                                                                 
 flatten (Flatten)           (None, 7680)              0

**12.3–12.4** — Code hoàn chỉnh: LeNet-5 Sequential + train trên CIFAR-10

In [2]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D, MaxPool2D

(trainX, trainY), (testX, testY) = tf.keras.datasets.cifar10.load_data()

model = Sequential([
          Input(shape=(32,32,3,)),
          Conv2D(6, (5,5), padding="same", activation="relu"),
          MaxPool2D(pool_size=(2,2)),
          Conv2D(16, (5,5), padding="same", activation="relu"),
          MaxPool2D(pool_size=(2, 2)),
          Conv2D(120, (5,5), padding="same", activation="relu"),
          Flatten(),
          Dense(units=84, activation="relu"),
          Dense(units=10, activation="softmax"),
      ])

model.summary()

model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics="acc")
history = model.fit(x=trainX, y=trainY, batch_size=256, epochs=10,
                    validation_data=(testX, testY))


170498071/170498071 [==============================] - 323s 2us/step
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 32, 32, 6)         456       
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 16, 16, 6)         0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 16, 16, 16)        2416      
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 8, 8, 16)          0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 8, 8, 120)         48120     
                                                   

2026-09-06 07:21:11.079644: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 153600000 exceeds 10% of free system memory.


Epoch 1/10


2026-09-06 07:21:14.857641: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 27033600 exceeds 10% of free system memory.
2026-09-06 07:21:14.871062: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 27033600 exceeds 10% of free system memory.
2026-09-06 07:21:14.967062: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 29184000 exceeds 10% of free system memory.


  2/196 [..............................] - ETA: 2:52 - loss: 90.5290 - acc: 0.0918 

2026-09-06 07:21:15.795268: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 27033600 exceeds 10% of free system memory.


196/196 [==============================] - 63s 306ms/step - loss: 5.2113 - acc: 0.3338 - val_loss: 1.5881 - val_acc: 0.4261
Epoch 2/10
196/196 [==============================] - 46s 234ms/step - loss: 1.4993 - acc: 0.4654 - val_loss: 1.4559 - val_acc: 0.4795
Epoch 3/10
196/196 [==============================] - 46s 235ms/step - loss: 1.3888 - acc: 0.5061 - val_loss: 1.3962 - val_acc: 0.5000
Epoch 4/10
196/196 [==============================] - 46s 234ms/step - loss: 1.3020 - acc: 0.5397 - val_loss: 1.3431 - val_acc: 0.5189
Epoch 5/10
196/196 [==============================] - 46s 234ms/step - loss: 1.2246 - acc: 0.5678 - val_loss: 1.3089 - val_acc: 0.5386
Epoch 6/10
196/196 [==============================] - 46s 234ms/step - loss: 1.1658 - acc: 0.5863 - val_loss: 1.2998 - val_acc: 0.5405
Epoch 7/10
196/196 [==============================] - 46s 233ms/step - loss: 1.1005 - acc: 0.6114 - val_loss: 1.2857 - val_acc: 0.5561
Epoch 8/10
196/196 [==============================] - 46s 237ms/st

**12.5–12.6** — LeNet-5 bằng Functional API

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.models import Model

input_layer = Input(shape=(32,32,3,))
x = Conv2D(6, (5,5), padding="same", activation="relu")(input_layer)
x = MaxPool2D(pool_size=(2,2))(x)
x = Conv2D(16, (5,5), padding="same", activation="relu")(x)
x = MaxPool2D(pool_size=(2, 2))(x)
x = Conv2D(120, (5,5), padding="same", activation="relu")(x)
x = Flatten()(x)
x = Dense(units=84, activation="relu")(x)
x = Dense(units=10, activation="softmax")(x)

model = Model(inputs=input_layer, outputs=x)

model.summary()


2026-09-06 07:47:04.737264: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:47:05.916901: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 07:47:05.917018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 07:47:06.140764: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 07:47:06.451616: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:47:06.457270: I tensorflow/core/platform/cpu_feature_guard.cc:1

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 32, 6)         456       
                                                                 
 max_pooling2d (MaxPooling2  (None, 16, 16, 6)         0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 16)        2416      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 8, 8, 16)          0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 8, 8, 120)         48120 

**12.7–12.9** — Residual block (ResNet) + train trên CIFAR-10 — chỉ làm được với Functional API

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, \
                                    MaxPool2D, Flatten, Dense
from tensorflow.keras.activations import relu
from tensorflow.keras.models import Model

# Release graphs and resources from the previous model before building this one.
tf.keras.backend.clear_session()

def residual_block(x, filters):
    # Store the input tensor to add it back after the convolution layers.
    identity = x
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=(1, 1), padding="same")(x)
    x = BatchNormalization()(x)
    x = relu(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = Add()([identity, x])
    x = relu(x)

    return x

(trainX, trainY), (testX, testY) = tf.keras.datasets.cifar10.load_data()

input_layer = Input(shape=(32,32,3,))
x = Conv2D(32, (3, 3), padding="same", activation="relu")(input_layer)
x = residual_block(x, 32)
x = Conv2D(64, (3, 3), strides=(2, 2), padding="same", activation="relu")(x)
x = residual_block(x, 64)
x = Conv2D(128, (3, 3), strides=(2, 2), padding="same", activation="relu")(x)
x = residual_block(x, 128)
x = Flatten()(x)
x = Dense(units=84, activation="relu")(x)
x = Dense(units=10, activation="softmax")(x)

model = Model(inputs=input_layer, outputs=x)
model.summary()

model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics="acc")

history = model.fit(x=trainX, y=trainY, batch_size=64, epochs=10,
                    validation_data=(testX, testY))


2026-09-06 07:57:37.731611: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:57:38.407961: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 07:57:38.408070: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 07:57:38.663607: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 07:57:38.989409: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:57:38.991209: I tensorflow/core/platform/cpu_feature_guard.cc:1

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 32, 32, 3)]          0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 32, 32, 32)           896       ['input_1[0][0]']             
                                                                                                  
 conv2d_1 (Conv2D)           (None, 32, 32, 32)           9248      ['conv2d[0][0]']              
                                                                                                  
 batch_normalization (Batch  (None, 32, 32, 32)           128       ['conv2d_1[0][0]']            
 Normalization)                                                                               

2026-09-06 07:57:49.269790: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 153600000 exceeds 10% of free system memory.
2026-09-06 07:57:52.016047: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 23592960 exceeds 10% of free system memory.
2026-09-06 07:57:52.068641: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 23592960 exceeds 10% of free system memory.
2026-09-06 07:57:52.106354: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 21528576 exceeds 10% of free system memory.
2026-09-06 07:57:52.106462: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 21528576 exceeds 10% of free system memory.


782/782 [==============================] - 397s 505ms/step - loss: 2.0960 - acc: 0.3294 - val_loss: 2.0154 - val_acc: 0.3071
Epoch 2/10
782/782 [==============================] - 394s 504ms/step - loss: 1.3727 - acc: 0.5159 - val_loss: 1.2225 - val_acc: 0.5885
Epoch 3/10
782/782 [==============================] - 393s 502ms/step - loss: 1.0312 - acc: 0.6380 - val_loss: 1.0826 - val_acc: 0.6262
Epoch 4/10
782/782 [==============================] - 387s 495ms/step - loss: 0.8613 - acc: 0.6921 - val_loss: 0.8886 - val_acc: 0.6906
Epoch 5/10
782/782 [==============================] - 392s 501ms/step - loss: 0.7511 - acc: 0.7322 - val_loss: 0.8483 - val_acc: 0.7143
Epoch 6/10
782/782 [==============================] - 393s 502ms/step - loss: 0.6533 - acc: 0.7665 - val_loss: 0.9400 - val_acc: 0.6864
Epoch 7/10
782/782 [==============================] - 386s 494ms/step - loss: 0.5651 - acc: 0.7978 - val_loss: 0.9586 - val_acc: 0.7010
Epoch 8/10
782/782 [==============================] - 392s 

**12.10–12.14** — LeNet-5 bằng subclassing keras.Model

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Flatten, Conv2D, MaxPool2D
from tensorflow.keras.models import Model

class LeNet5(tf.keras.Model):
  def __init__(self):
    super(LeNet5, self).__init__()
    #creating layers in initializer
    self.conv1 = Conv2D(6, (5,5), padding="same", activation="relu")
    self.max_pool2x2 = MaxPool2D(pool_size=(2,2))
    self.conv2 = Conv2D(16, (5,5), padding="same", activation="relu")
    self.conv3 = Conv2D(120, (5,5), padding="same", activation="relu")
    self.flatten = Flatten()
    self.fc2 = Dense(units=84, activation="relu")
    self.fc3=Dense(units=10, activation="softmax")

  def call(self, input_tensor):
    # don't add layers here, need to create the layers in initializer,
    # otherwise you will get the tf.Variable can only be created once error
    x = self.conv1(input_tensor)
    x = self.max_pool2x2(x)
    x = self.conv2(x)
    x = self.max_pool2x2(x)
    x = self.conv3(x)
    x = self.flatten(x)
    x = self.fc2(x)
    x = self.fc3(x)
    return x

input_layer = Input(shape=(32,32,3,))
x = LeNet5()(input_layer)
model = Model(inputs=input_layer, outputs=x)
model.summary(expand_nested=True)


2026-09-06 09:46:31.832387: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 09:46:43.071254: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 09:46:43.071352: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 09:46:44.583192: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 09:46:47.435542: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 09:46:47.436674: I tensorflow/core/platform/cpu_feature_guard.cc:1

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 le_net5 (LeNet5)            (None, 10)                697046    
|¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯¯|
| conv2d (Conv2D)            multiple                  456      |
|                                                               |
| max_pooling2d (MaxPooling  multiple                  0        |
| 2D)                                                           |
|                                                               |
| conv2d_1 (Conv2D)          multiple                  2416     |
|                                                               |
| conv2d_2 (Conv2D)          multiple                  48120    |
|                                                            